# CICIDS2017 - Preprocessing Pipeline
Output: cleaned CSV + final feature list

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = r'D:\Academic\ML\ModelTraining\dataset\MachineLearningCVE'
OUTPUT_CSV = r'D:\Academic\ML\ModelTraining\Preprocessing\cicids2017_preprocessed.csv'
FEATURE_LIST_TXT = r'D:\Academic\ML\ModelTraining\Preprocessing\final_features.txt'

## 1. Load & Merge

In [2]:
import os
dfs = []
for f in sorted(os.listdir(DATA_DIR)):
    if not f.endswith('.csv'):
        continue
    df_temp = pd.read_csv(os.path.join(DATA_DIR, f), low_memory=False)
    df_temp.columns = df_temp.columns.str.strip()
    dfs.append(df_temp)
    print(f'{f}: {df_temp.shape}')

df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined: {df.shape}')

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (225745, 79)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (286467, 79)
Friday-WorkingHours-Morning.pcap_ISCX.csv: (191033, 79)
Monday-WorkingHours.pcap_ISCX.csv: (529918, 79)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (288602, 79)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (170366, 79)
Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 79)
Wednesday-workingHours.pcap_ISCX.csv: (692703, 79)

Combined: (2830743, 79)


## 2. Label Grouping — 5 Classes Only

In [3]:
label_col = ' Label' if ' Label' in df.columns else 'Label'
df[label_col] = df[label_col].str.strip()

label_map = {
    'BENIGN': 'Normal Traffic',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'DDoS': 'DDoS',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'PortScan': 'Port Scan',
    # Drop these
    'Bot': None,
    'Web Attack \ufffd Brute Force': None,
    'Web Attack \ufffd XSS': None,
    'Web Attack \ufffd Sql Injection': None,
    'Infiltration': None,
    'Heartbleed': None,
}

unmapped = set(df[label_col].unique()) - set(label_map.keys())
if unmapped:
    print(f'WARNING unmapped: {unmapped}')

df['Attack Type'] = df[label_col].map(label_map)
df = df[df['Attack Type'].notna()].reset_index(drop=True)
df = df.drop(columns=[label_col])
print(df['Attack Type'].value_counts())
print(f'Shape: {df.shape}')

Attack Type
Normal Traffic    2273097
DoS                252661
Port Scan          158930
DDoS               128027
Brute Force         13835
Name: count, dtype: int64
Shape: (2826550, 79)


## 3. Drop Constant Columns

In [4]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

constant_cols = [c for c in numeric_cols if df[c].nunique() <= 1]
print(f'Dropping constant columns: {constant_cols}')
df = df.drop(columns=constant_cols)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

Dropping constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


## 4. Handle Inf / NaN

In [5]:
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
before = len(df)
df = df.dropna(subset=numeric_cols).reset_index(drop=True)
print(f'Dropped {before - len(df)} rows with NaN/inf. Remaining: {len(df):,}')

Dropped 2857 rows with NaN/inf. Remaining: 2,823,693


## 5. Drop Duplicate Rows

In [6]:
before = len(df)
df = df.drop_duplicates(subset=numeric_cols).reset_index(drop=True)
print(f'Dropped {before - len(df)} duplicates. Remaining: {len(df):,}')
print(df['Attack Type'].value_counts())

Dropped 307730 duplicates. Remaining: 2,515,963
Attack Type
Normal Traffic    2094458
DoS                193647
DDoS               128014
Port Scan           90694
Brute Force          9150
Name: count, dtype: int64


## 6. Drop Correlated Features

In [7]:
corr_drop = [
    'Fwd IAT Total', 'Subflow Fwd Packets', 'Subflow Bwd Packets',
    'Subflow Fwd Bytes', 'Subflow Bwd Bytes', 'Avg Fwd Segment Size',
    'Avg Bwd Segment Size', 'Fwd PSH Flags', 'Fwd URG Flags',
    'Fwd Header Length.1', 'ECE Flag Count', 'Average Packet Size',
    'Fwd IAT Max', 'Fwd Packets/s', 'Idle Max', 'Idle Min'
]

# Also drop Down/Up Ratio if present (near-zero variance in practice)
corr_drop.append('Down/Up Ratio')

existing = [c for c in corr_drop if c in df.columns]
missing_cols = [c for c in corr_drop if c not in df.columns]
if missing_cols:
    print(f'Already absent (likely constant): {missing_cols}')

df = df.drop(columns=existing)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Remaining features: {len(numeric_cols)}')
print(numeric_cols)

Remaining features: 53
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean

## 7. Kruskal-Wallis Test — Drop Statistically Irrelevant Features

In [8]:
# Sample for speed
sample = df.sample(min(50000, len(df)), random_state=42)
groups = [sample[sample['Attack Type'] == c][numeric_cols].values 
          for c in sample['Attack Type'].unique()]

kw_results = {}
for col in numeric_cols:
    col_groups = [sample[sample['Attack Type'] == c][col].values 
                  for c in sample['Attack Type'].unique()]
    try:
        stat, p = kruskal(*col_groups)
        kw_results[col] = p
    except:
        kw_results[col] = 1.0

kw_series = pd.Series(kw_results).sort_values(ascending=False)
kw_drop = kw_series[kw_series > 0.05].index.tolist()
print(f'Kruskal-Wallis drop (p > 0.05): {kw_drop}')

if kw_drop:
    df = df.drop(columns=kw_drop)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Remaining features after KW: {len(numeric_cols)}')

Kruskal-Wallis drop (p > 0.05): ['CWE Flag Count', 'RST Flag Count']
Remaining features after KW: 51


## 8. Random Forest Feature Importance

In [9]:
sample = df.sample(min(50000, len(df)), random_state=42)
le = LabelEncoder()
y_sample = le.fit_transform(sample['Attack Type'])
X_sample = sample[numeric_cols]

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = pd.Series(rf.feature_importances_, index=numeric_cols).sort_values(ascending=False)
print(importances)

# Drop features with near-zero importance (bottom 10% threshold)
threshold = importances.quantile(0.10)
rf_drop = importances[importances < threshold].index.tolist()
print(f'\nRF drop (bottom 10%): {rf_drop}')

if rf_drop:
    df = df.drop(columns=rf_drop)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Final feature count: {len(numeric_cols)}')
print(numeric_cols)

Bwd Packet Length Mean         0.071683
Packet Length Std              0.069747
Bwd Packet Length Std          0.067656
Packet Length Variance         0.062067
Max Packet Length              0.047904
Fwd Packet Length Max          0.045052
Total Length of Bwd Packets    0.044858
Packet Length Mean             0.041438
Total Length of Fwd Packets    0.035506
Bwd Packet Length Max          0.031479
Fwd Header Length              0.030221
Fwd Packet Length Mean         0.025126
Idle Mean                      0.024834
Flow IAT Max                   0.024073
Flow IAT Std                   0.023863
Total Fwd Packets              0.022983
act_data_pkt_fwd               0.022246
Fwd IAT Std                    0.021828
Destination Port               0.021698
Init_Win_bytes_backward        0.021074
Flow Duration                  0.019579
Bwd Header Length              0.019506
min_seg_size_forward           0.018373
PSH Flag Count                 0.017526
Fwd IAT Mean                   0.017471


## 9. Save Output

In [10]:
# Final column order: features first, Attack Type last
final_cols = numeric_cols + ['Attack Type']
df = df[final_cols]

print(f'Final shape: {df.shape}')
print(f'\nClass distribution:')
print(df['Attack Type'].value_counts())

# Save CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved: {OUTPUT_CSV}')

# Save feature list
with open(FEATURE_LIST_TXT, 'w') as f:
    for col in numeric_cols:
        f.write(col + '\n')
print(f'Saved feature list: {FEATURE_LIST_TXT}')
print('\nFinal features:')
for i, col in enumerate(numeric_cols):
    print(f'  {i+1:2d}. {col}')

Final shape: (2515963, 47)

Class distribution:
Attack Type
Normal Traffic    2094458
DoS                193647
DDoS               128014
Port Scan           90694
Brute Force          9150
Name: count, dtype: int64

Saved: D:\Academic\ML\ModelTraining\Preprocessing\cicids2017_preprocessed.csv
Saved feature list: D:\Academic\ML\ModelTraining\Preprocessing\final_features.txt

Final features:
   1. Destination Port
   2. Flow Duration
   3. Total Fwd Packets
   4. Total Backward Packets
   5. Total Length of Fwd Packets
   6. Total Length of Bwd Packets
   7. Fwd Packet Length Max
   8. Fwd Packet Length Min
   9. Fwd Packet Length Mean
  10. Fwd Packet Length Std
  11. Bwd Packet Length Max
  12. Bwd Packet Length Min
  13. Bwd Packet Length Mean
  14. Bwd Packet Length Std
  15. Flow Bytes/s
  16. Flow Packets/s
  17. Flow IAT Mean
  18. Flow IAT Std
  19. Flow IAT Max
  20. Flow IAT Min
  21. Fwd IAT Mean
  22. Fwd IAT Std
  23. Fwd IAT Min
  24. Bwd IAT Total
  25. Bwd IAT Mean
  26.